# STEP 1 - Day-1 Gate: does the zero-width attack actually work?

**Before running, set these in the right-hand panel:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (P100 also fine - only 1 GPU is used) |
| Internet | **ON** (needs phone-verified Kaggle account) |
| Add Data | search `bdshs` -> add **naurosromim/bdshs** |

Then: **Run All**. Expect ~20-30 min. Copy the SUMMARY block at the end back to me.

This answers two questions:
1. Does inserting invisible characters make a Bangla hate-speech classifier miss hate?
2. Does BanglaBERT's own official normalizer stop it? (spoiler: ZWSP yes, ZWNJ/ZWJ no)


In [ ]:
# 1. Setup
!pip install -q "emoji==1.7.0" regex ftfy
!pip install -q git+https://github.com/csebuetnlp/normalizer

import os, sys, json, random, unicodedata, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "|", torch.cuda.get_device_name(0) if DEV=="cuda" else "")

In [ ]:
# 2. Attack + defense library (self-contained)
ZWNJ, ZWJ, ZWSP = "\u200c", "\u200d", "\u200b"
HASANT = "\u09cd"
ZW_ALL = (ZWNJ, ZWJ, ZWSP)

def is_bangla(ch): return 0x0980 <= ord(ch) <= 0x09FF

def inert_positions(text):
    """Both neighbours must be Bangla BASE letters (category Lo) -> rendering
    is guaranteed unchanged (no combining mark or hasant interaction)."""
    out=[]
    for i in range(1,len(text)):
        a,b = text[i-1], text[i]
        if not (is_bangla(a) and is_bangla(b)): continue
        if unicodedata.category(a)!="Lo" or unicodedata.category(b)!="Lo": continue
        out.append(i)
    return out

def attack(text, budget=1, char=ZWNJ, seed=0):
    pos = inert_positions(text)
    if not pos: return text
    rng = random.Random(seed)
    for i in sorted(rng.sample(pos, min(budget,len(pos))), reverse=True):
        text = text[:i] + char + text[i:]
    return text

def defend_strip_all(t):
    for c in ZW_ALL: t = t.replace(c,"")
    return t

def defend_position_aware(t):
    """Drop zero-width chars only where orthographically inert; keep the
    hasant-adjacent ones that actually control conjunct formation."""
    out=[]
    for i,ch in enumerate(t):
        if ch in (ZWNJ,ZWJ):
            prev = t[i-1] if i>0 else ""
            nxt  = t[i+1] if i+1<len(t) else ""
            if prev==HASANT or nxt==HASANT: out.append(ch)
            continue
        if ch==ZWSP: continue
        out.append(ch)
    return "".join(out)

# sanity check
_s = "\u09a4\u09c1\u0987 \u098f\u0995\u099f\u09be \u09ac\u09be\u099c\u09c7 \u09b2\u09cb\u0995"
_a = attack(_s, 2)
print("renders identically :", _s == _a.replace(ZWNJ,""))
print("defense restores    :", defend_position_aware(_a) == _s)

In [ ]:
# 3. THE HEADLINE TEST - what survives BanglaBERT's OFFICIAL normalizer?
rows=[]
base = "\u09a4\u09c1\u0987 \u098f\u0995\u099f\u09be \u09ac\u09be\u099c\u09c7 \u09b2\u09cb\u0995"
for name,ch in [("ZWSP",ZWSP),("ZWNJ",ZWNJ),("ZWJ",ZWJ),("SHY","\u00ad"),("WJ","\u2060")]:
    atk = attack(base, 1, char=ch, seed=1)
    if atk == base: atk = base[:3] + ch + base[3:]
    survives = ch in normalize(atk)
    rows.append({"char":name,"codepoint":"U+%04X"%ord(ch),
                 "survives_official_normalizer":survives})
zw_table = pd.DataFrame(rows)
print(zw_table.to_string(index=False))
zw_table.to_csv("/kaggle/working/normalizer_survival.csv", index=False)

In [ ]:
# 4. Load BD-SHS and auto-detect schema
import glob
paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV files found:"); [print("  ", p) for p in paths]

def pick(fnames, key):
    for p in fnames:
        if key in os.path.basename(p).lower(): return p
    return None

tr_p, te_p = pick(paths,"train"), pick(paths,"test")
train = pd.read_csv(tr_p); test = pd.read_csv(te_p)
print("\ntrain shape", train.shape, "| test shape", test.shape)
print("columns:", list(train.columns))

# text col = object col with longest mean length; label col = binary numeric
obj = [c for c in train.columns if train[c].dtype==object]
TEXT = max(obj, key=lambda c: train[c].astype(str).str.len().mean())
cand = [c for c in train.columns if c!=TEXT and train[c].dropna().nunique()==2]
LABEL = cand[0]
print("\n>>> TEXT column :", TEXT)
print(">>> LABEL column:", LABEL, "| distribution:", dict(train[LABEL].value_counts()))

In [ ]:
# 5. Preprocess with the OFFICIAL pipeline (the realistic deployment setting)
def prep(df):
    df = df[[TEXT,LABEL]].dropna().copy()
    df.columns = ["text","label"]
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)
    return df[df["text"].str.strip().str.len()>0].reset_index(drop=True)

train, test = prep(train), prep(test)
MODEL="csebuetnlp/banglabert"; MAXLEN=128
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(texts):
    e = tok([normalize(t) for t in texts], truncation=True, padding="max_length",
            max_length=MAXLEN, return_tensors="pt")
    return e["input_ids"], e["attention_mask"]

Xtr, Mtr = encode(train["text"].tolist()); ytr = torch.tensor(train["label"].values)
print("train:", Xtr.shape, "| test:", len(test))

In [ ]:
# 6. Fine-tune BanglaBERT  (~10-20 min on T4)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
dl = DataLoader(TensorDataset(Xtr,Mtr,ytr), batch_size=32, shuffle=True)
EPOCHS=2
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-5,
            total_steps=EPOCHS*len(dl), pct_start=0.1)
scaler = torch.cuda.amp.GradScaler()

model.train()
for ep in range(EPOCHS):
    tot=0
    for step,(ids,msk,lab) in enumerate(dl):
        ids,msk,lab = ids.to(DEV),msk.to(DEV),lab.to(DEV)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(input_ids=ids, attention_mask=msk, labels=lab).loss
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot+=loss.item()
        if step%150==0: print(f"  ep{ep} step{step}/{len(dl)} loss {loss.item():.4f}")
    print(f"epoch {ep} mean loss {tot/len(dl):.4f}")
model.eval(); print("done")

In [ ]:
# 7. Predict helper
@torch.no_grad()
def predict(texts, bs=128):
    out=[]
    for i in range(0,len(texts),bs):
        chunk=[normalize(t) for t in texts[i:i+bs]]
        e=tok(chunk,truncation=True,padding="max_length",max_length=MAXLEN,return_tensors="pt")
        with torch.cuda.amp.autocast():
            lg=model(input_ids=e["input_ids"].to(DEV),
                     attention_mask=e["attention_mask"].to(DEV)).logits
        out.append(lg.float().argmax(-1).cpu())
    return torch.cat(out).numpy()

te_txt = test["text"].tolist(); te_y = test["label"].values
clean_pred = predict(te_txt)
base_acc = accuracy_score(te_y, clean_pred); base_f1 = f1_score(te_y, clean_pred)
print(f"CLEAN baseline -> acc {base_acc:.4f} | macro-ish F1 {base_f1:.4f}")

HATE = int(train["label"].value_counts().idxmin()) if False else 1
hit = np.where((te_y==HATE) & (clean_pred==HATE))[0]
print(f"correctly-detected hate comments (attack targets): {len(hit)}")

In [ ]:
# 8. ATTACK SWEEP - evasion rate vs budget vs character
sub = hit[:2000]
sub_txt = [te_txt[i] for i in sub]
res=[]
for ch_name, ch in [("ZWNJ",ZWNJ),("ZWJ",ZWJ),("ZWSP",ZWSP)]:
    for budget in [1,2,3,5]:
        atk=[attack(t,budget=budget,char=ch,seed=SEED) for t in sub_txt]
        changed = sum(a!=t for a,t in zip(atk,sub_txt))
        p = predict(atk)
        evaded = float((p!=HATE).mean())
        res.append({"char":ch_name,"budget":budget,
                    "evasion_rate":round(evaded,4),
                    "texts_modified":changed})
        print(f"{ch_name:5} budget={budget} -> evasion {evaded:.2%}  (modified {changed}/{len(sub_txt)})")
atk_df=pd.DataFrame(res); atk_df.to_csv("/kaggle/working/attack_sweep.csv",index=False)

In [ ]:
# 9. Is functional ZWNJ/ZWJ common in real Bangla text?
def zw_stats(texts):
    d=zw=fn=inert=0
    for t in texts:
        h=False
        for i,c in enumerate(t):
            if c not in (ZWNJ,ZWJ): continue
            h=True; zw+=1
            prev=t[i-1] if i>0 else ""; nxt=t[i+1] if i+1<len(t) else ""
            if prev==HASANT or nxt==HASANT: fn+=1
            else: inert+=1
        d+=h
    return dict(n_docs=len(texts),docs_with_zw=d,total_zw=zw,functional=fn,inert=inert)

corpus_stats = zw_stats(pd.concat([train["text"],test["text"]]).tolist())
print(json.dumps(corpus_stats, indent=2))
json.dump(corpus_stats, open("/kaggle/working/corpus_zw_stats.json","w"), indent=2)

In [ ]:
# 10. SUMMARY  <-- copy this whole block back
print("="*64); print("STEP 1 SUMMARY"); print("="*64)
print(f"train/test sizes      : {len(train)} / {len(test)}")
print(f"clean baseline acc/F1 : {base_acc:.4f} / {base_f1:.4f}")
print(f"attack targets        : {len(hit)}")
print("\n-- normalizer survival --"); print(zw_table.to_string(index=False))
print("\n-- evasion rates --");       print(atk_df.to_string(index=False))
print("\n-- corpus zero-width usage --"); print(json.dumps(corpus_stats))
print("="*64)